# Home Matching System - Comprehensive Testing & Performance Analysis

This notebook provides comprehensive testing and performance analysis of the home matching system with visualizations and metrics.

## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import time
import logging
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Import our home matching system
from home_matching import (
    find_best_matches, 
    score_single_match,
    compare_homes_for_user,
    create_sample_user,
    create_sample_home,
    get_system_info
)

# Set up logging
logging.basicConfig(level=logging.WARNING)  # Reduce noise
logger = logging.getLogger(__name__)

print("✅ All imports successful!")
print(f"📊 Matplotlib version: {plt.matplotlib.__version__}")
print(f"🐼 Pandas version: {pd.__version__}")
print(f"🔢 NumPy version: {np.__version__}")

ModuleNotFoundError: No module named 'torch'

## System Information

In [ ]:
# Get and display system information
system_info = get_system_info()
print("🏠 Home Matching System Information:")
for key, value in system_info.items():
    print(f"  {key}: {value}")

## Data Generation for Testing

In [ ]:
def create_diverse_users(n_users: int = 10) -> List[Dict[str, Any]]:
    """Create diverse user profiles for testing."""
    users = []
    
    # Define user archetypes
    archetypes = [
        {
            'lifestyle': 'Young Professional',
            'budget_range': (300000, 600000),
            'bedrooms': 2,
            'bathrooms': 2,
            'min_sqft': 1200,
            'must_have': ['garage'],
            'nice_to_have': ['gym', 'pool']
        },
        {
            'lifestyle': 'Growing Family',
            'budget_range': (500000, 900000),
            'bedrooms': 4,
            'bathrooms': 3,
            'min_sqft': 2200,
            'must_have': ['yard', 'garage'],
            'nice_to_have': ['pool', 'playground']
        },
        {
            'lifestyle': 'Empty Nesters',
            'budget_range': (400000, 800000),
            'bedrooms': 3,
            'bathrooms': 2.5,
            'min_sqft': 1800,
            'must_have': ['garage'],
            'nice_to_have': ['garden', 'wine_cellar']
        },
        {
            'lifestyle': 'First-time Buyer',
            'budget_range': (250000, 450000),
            'bedrooms': 2,
            'bathrooms': 1.5,
            'min_sqft': 1000,
            'must_have': [],
            'nice_to_have': ['garage', 'yard']
        }
    ]
    
    for i in range(n_users):
        archetype = archetypes[i % len(archetypes)]
        
        # Add some randomness to budget
        budget_min, budget_max = archetype['budget_range']
        budget_variation = np.random.uniform(0.9, 1.1)
        
        user = {
            'user_id': f'test_user_{i+1:03d}',
            'preferences': {
                'budget_min': int(budget_min * budget_variation),
                'budget_max': int(budget_max * budget_variation),
                'preferred_bedrooms': archetype['bedrooms'],
                'preferred_bathrooms': archetype['bathrooms'],
                'min_sqft': archetype['min_sqft'],
                'lifestyle': archetype['lifestyle'],
                'must_have_amenities': archetype['must_have'],
                'nice_to_have_amenities': archetype['nice_to_have'],
                'housing_type': 'house',
                'max_commute_minutes': np.random.randint(20, 45)
            }
        }
        users.append(user)
    
    return users

In [ ]:
def create_diverse_homes(n_homes: int = 50) -> List[Dict[str, Any]]:
    """Create diverse home listings for testing."""
    homes = []
    
    # Define home characteristics
    neighborhoods = ['Downtown', 'Suburbs', 'Upscale', 'Affordable', 'Trendy']
    home_types = ['house', 'townhouse', 'condo']
    styles = ['modern', 'traditional', 'contemporary', 'colonial', 'ranch']
    
    for i in range(n_homes):
        # Generate realistic home characteristics
        bedrooms = np.random.choice([2, 3, 4, 5], p=[0.2, 0.4, 0.3, 0.1])
        bathrooms = bedrooms - 0.5 + np.random.choice([0, 0.5, 1])
        sqft = int(np.random.normal(bedrooms * 600 + 800, 300))
        sqft = max(1000, sqft)  # Minimum size
        
        # Price based on size and neighborhood
        neighborhood = np.random.choice(neighborhoods)
        base_price_per_sqft = {
            'Downtown': 350,
            'Suburbs': 250,
            'Upscale': 450,
            'Affordable': 200,
            'Trendy': 400
        }[neighborhood]
        
        price = int(sqft * base_price_per_sqft * np.random.uniform(0.8, 1.2))
        
        # Amenities based on price range
        amenities = ['garage'] if price > 300000 else []
        if price > 400000:
            amenities.extend(['yard'])
        if price > 600000:
            amenities.extend(np.random.choice(['pool', 'gym', 'wine_cellar'], size=1).tolist())
        
        home = {
            'home_id': f'test_home_{i+1:03d}',
            'address': f'{100 + i} Test Street, {neighborhood}, ST 12345',
            'price': price,
            'bedrooms': bedrooms,
            'bathrooms': bathrooms,
            'sqft': sqft,
            'home_type': np.random.choice(home_types),
            'style': np.random.choice(styles),
            'neighborhood': neighborhood,
            'amenities': amenities,
            'has_garage': 'garage' in amenities,
            'has_yard': 'yard' in amenities,
            'has_pool': 'pool' in amenities,
            'pet_friendly': np.random.choice([True, False]),
            'commute_minutes': np.random.randint(15, 50),
            'description': f'Beautiful {styles[i % len(styles)]} {home_types[i % len(home_types)]} in {neighborhood}.'
        }
        homes.append(home)
    
    return homes

# Generate test data
print("🏗️ Generating test data...")
test_users = create_diverse_users(10)
test_homes = create_diverse_homes(50)

print(f"✅ Created {len(test_users)} diverse users and {len(test_homes)} diverse homes")
print(f"📊 Price range: ${min(h['price'] for h in test_homes):,} - ${max(h['price'] for h in test_homes):,}")
print(f"🏠 Bedrooms range: {min(h['bedrooms'] for h in test_homes)} - {max(h['bedrooms'] for h in test_homes)}")